In [1]:
import os
import json
import re
from flaml.autogen.math_utils import eval_math_responses, get_answer
from flaml import oai
from utils import math_type_mapping, load_config, setup_proxy
import time

/home/javad/miniconda3/envs/mathchat/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def verify_false_questions(log_file, config_list, output_file="false_verified.json", use_cache=True):
    """Verify questions marked as False in log.txt"""
    
    with open(log_file, 'r') as f:
        content = f.read()
    
    sections = re.split(r'id : is_correct \$ ans \$ correct_ans \$ corrected_ans \$ round \$', content)
    
    questions = {}
    for section in sections:
        if not section.strip():
            continue
        
        current_category = None
        for cat in sorted(math_type_mapping.keys(), key=len, reverse=True):
            if f"{cat} Accuracy:" in section:
                current_category = cat
                break
        
        if not current_category:
            continue
        
        lines = section.strip().split('\n')
        for line in lines:
            line = line.strip()
            if ' : ' in line and '$' in line:
                parts = line.split(' : ')
                if len(parts) != 2:
                    continue
                
                q_id = parts[0].strip()
                values = [v.strip() for v in parts[1].split('$')]
                
                if len(values) < 3:
                    continue
                
                is_correct = values[0].strip() == 'True'
                ans = values[1].strip()
                correct_ans = values[2].strip()
                
                if q_id.startswith('question_') and not is_correct:
                    if current_category not in questions:
                        questions[current_category] = []
                    if not any(q[0] == q_id for q in questions[current_category]):
                        questions[current_category].append((q_id, ans, correct_ans))
    
    if os.path.exists(output_file):
        with open(output_file, 'r') as f:
            existing = json.load(f)
    else:
        existing = {"questions": {}, "count": {}}
    
    for category, q_list in questions.items():
        for q_id, ans, correct_ans in q_list:
            if ans == "None" or correct_ans == "None":
                continue
            
            if category in existing["questions"] and q_id in existing["questions"][category]:
                continue
            
            eval_result = eval_math_responses(['\\boxed{' + ans + '}'], '\\boxed{' + correct_ans + '}')
            
            if eval_result["success_vote"]:
                if category not in existing["questions"]:
                    existing["questions"][category] = []
                    existing["count"][category] = 0
                existing["questions"][category].append(q_id)
                existing["count"][category] += 1
                with open(output_file, 'w') as f:
                    json.dump(existing, f, indent=2)
                continue
            
            verify_prompt = f"""You are a math expert. Compare these two answers and determine if they are mathematically equivalent.

Model answer: {ans}
Correct answer: {correct_ans}

Respond ONLY with 'YES' or 'NO'."""

            config = {
                "messages": [
                    {"role": "system", "content": "You are a math verification expert."},
                    {"role": "user", "content": verify_prompt}
                ],
                "n": 1,
                "temperature": 0.1
            }

            while True:
                try:
                    raw_response = oai.ChatCompletion.create(
                        config_list=config_list,
                        **config,
                        use_cache=use_cache
                    )

                    result = oai.ChatCompletion.extract_text(raw_response)[0]
                    llm_verified = "YES" in result.upper()

                    if llm_verified:
                        if category not in existing["questions"]:
                            existing["questions"][category] = []
                            existing["count"][category] = 0
                        existing["questions"][category].append(q_id)
                        existing["count"][category] += 1
                    
                    with open(output_file, 'w') as f:
                        json.dump(existing, f, indent=2)
                        
                    break  # Success

                except Exception as e:
                    msg = str(e)
                    print(msg)

                    if (
                        "list' object has no attribute 'get'" in msg
                        or "Internal error encountered" in msg
                    ):
                        print("Retrying after transient Google/FLAML/server error...")
                        time.sleep(4)
                        continue
                    if (
                        "Timeout" in msg
                    ):
                        print("Timeout, Retrying ...")
                        time.sleep(4)
                        continue

                    error_file = output_file.replace('.json', '_errors.json')

                    if os.path.exists(error_file):
                        with open(error_file, 'r') as f:
                            errors = json.load(f)
                    else:
                        errors = []

                    errors.append({
                        "question": q_id,
                        "category": category,
                        "error": msg,
                        "ans": ans,
                        "correct_ans": correct_ans
                    })

                    with open(error_file, 'w') as f:
                        json.dump(errors, f, indent=2)

                    llm_verified = False
                    break               
    
    print(f"Total false questions verified: {sum(existing['count'].values())}")


In [ ]:
def verify_true_questions(log_file, output_file="true_not_equal.json"):
    """Find True questions where ans != correct_ans (no LLM call)"""
    
    with open(log_file, 'r') as f:
        content = f.read()
    
    sections = re.split(r'id : is_correct \$ ans \$ correct_ans \$ corrected_ans \$ round \$', content)
    questions = {}
    
    for section in sections:
        if not section.strip():
            continue
        
        current_category = None
        for cat in sorted(math_type_mapping.keys(), key=len, reverse=True):
            if f"{cat} Accuracy:" in section:
                current_category = cat
                break
        
        if not current_category:
            continue
        
        lines = section.strip().split('\nz')
        for line in lines:
            line = line.strip()
            if ' : ' in line and '$' in line:
                parts = line.split(' : ')
                if len(parts) != 2:
                    continue
                
                q_id = parts[0].strip()
                values = [v.strip() for v in parts[1].split('$')]
                
                if len(values) < 3:
                    continue
                
                is_correct = values[0].strip() == 'True'
                ans = values[1].strip()
                correct_ans = values[2].strip()
                if q_id.startswith('question_') and is_correct:
                    eval_result = eval_math_responses(['\\boxed{' + ans + '}'], '\\boxed{' + correct_ans + '}')
                    
                    if not eval_result["success_vote"]:
                        if current_category not in questions:
                            questions[current_category] = []
                        if not any(q[0] == q_id for q in questions[current_category]):
                            questions[current_category].append((q_id, ans, correct_ans))
    
    if os.path.exists(output_file):
        with open(output_file, 'r') as f:
            existing = json.load(f)
    else:
        existing = {"questions": {}, "count": {}}
    a = set()
    for category, q_list in questions.items():
        a.add(category)
        for q_id, ans, correct_ans in q_list:
            if ans == "None" or correct_ans == "None":
                continue
            
            if category in existing["questions"] and q_id in existing["questions"][category]:
                continue
            
            if category not in existing["questions"]:
                existing["questions"][category] = []
                existing["count"][category] = 0
            existing["questions"][category].append(q_id)
            existing["count"][category] += 1
            
    with open(output_file, 'w') as f:
        json.dump(existing, f, indent=2)
    
    print(f"Total true not equal questions: {sum(existing['count'].values())}")
    print(a)

In [ ]:
verify_true_questions(log_file="/home/javad/Desktop/supernova/result_level_5_gemma_200_TL_1s0_user_v3.9python_token_limitation_t1/log.txt")

Total true not equal questions: 198
{'Prealgebra', 'Intermediate Algebra', 'Counting & Probability', 'Number Theory', 'Geometry', 'Precalculus', 'Algebra'}


In [3]:
config_list = load_config('./OAI_CONFIG_LIST/OAI_CONFIG_LIST gemini.json')
setup_proxy()

yes
no


In [30]:
verify_false_questions(log_file="/home/javad/Desktop/supernova/result_level_5_gemma_200_TL_1s0_user_v3.9python_token_limitation_t1/log.txt", config_list=config_list)

'list' object has no attribute 'get'
Retrying after transient Google/FLAML/server error...
'list' object has no attribute 'get'
Retrying after transient Google/FLAML/server error...
'list' object has no attribute 'get'
Retrying after transient Google/FLAML/server error...
'list' object has no attribute 'get'
Retrying after transient Google/FLAML/server error...
'list' object has no attribute 'get'
Retrying after transient Google/FLAML/server error...
'list' object has no attribute 'get'
Retrying after transient Google/FLAML/server error...
'list' object has no attribute 'get'
Retrying after transient Google/FLAML/server error...
'list' object has no attribute 'get'
Retrying after transient Google/FLAML/server error...
'list' object has no attribute 'get'
Retrying after transient Google/FLAML/server error...
'list' object has no attribute 'get'
Retrying after transient Google/FLAML/server error...
'list' object has no attribute 'get'
Retrying after transient Google/FLAML/server error...